# Bank of America - Machine Learning Predictive Models

## Objectives

This notebook develops predictive models for consumer complaint analysis:

### 1. Resolution Time Prediction
* **Goal**: Predict how many days it will take to resolve a complaint
* **Use Case**: Resource planning, SLA management, customer expectations
* **Features**: Product, issue, channel, state, historical patterns
* **Model**: Gradient Boosting Regressor

### 2. Issue Classification
* **Goal**: Auto-categorize complaint issues from text
* **Use Case**: Automated routing, trend detection, early warning
* **Features**: Product, sub-product, state, submission channel
* **Model**: Random Forest Classifier

### 3. Monetary Relief Prediction
* **Goal**: Predict likelihood of monetary relief outcome
* **Use Case**: Cost forecasting, case prioritization
* **Features**: Product, issue, response time, state
* **Model**: Logistic Regression

---

## Data Source
* **Silver Layer**: `Bank_of_America.Bank_of_America_Silver.consumer_complaints`
* **Training Data**: 62,516 historical complaints with outcomes
* **Features**: Product, issue, channel, state, temporal patterns
* **Target Variables**: response_time_days, issue category, resolution_status

---

## MLflow Tracking
All experiments logged to MLflow for reproducibility and model registry.

In [0]:
# =============================================================================
# SETUP AND DATA LOADING
# =============================================================================
# This cell imports necessary libraries and loads complaint data from the 
# Silver layer for machine learning model training

# Import data manipulation libraries
import pandas as pd  # For dataframe operations and data preprocessing
import numpy as np   # For numerical operations and array handling
from pyspark.sql import functions as F  # For Spark SQL operations (if needed)

# Import scikit-learn components for machine learning
from sklearn.model_selection import train_test_split  # Split data into train/test sets
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier  # ML algorithms
from sklearn.linear_model import LogisticRegression  # Classification algorithm
from sklearn.preprocessing import LabelEncoder, StandardScaler  # Data encoding and scaling
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, accuracy_score, roc_auc_score  # Model evaluation metrics

# Import MLflow for experiment tracking and model registry
import mlflow
import mlflow.sklearn

# Load data from the Silver layer table
# Silver layer contains cleaned, validated individual complaint records with enrichments
df_spark = spark.table("Bank_of_America.Bank_of_America_Silver.consumer_complaints")

# Convert Spark DataFrame to Pandas DataFrame for scikit-learn compatibility
# Note: For very large datasets (>1M rows), consider sampling or using distributed ML (e.g., Spark MLlib)
df = df_spark.toPandas()

# Display basic information about the loaded dataset
print(f"Loaded {len(df)} records from Silver layer")
print(f"\nDataset shape: {df.shape}")  # Shows (rows, columns)
print(f"\nColumns: {df.columns.tolist()}")  # Lists all available features
print(f"\nSample data:")  # Preview first few rows to understand data structure
display(df.head())

In [0]:
# =============================================================================
# FEATURE ENGINEERING
# =============================================================================
# Transform raw data into ML-ready features by creating temporal features,
# handling missing values, and encoding categorical variables

print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

# -------------------------------------------------------------------------
# 1. CREATE TEMPORAL FEATURES
# -------------------------------------------------------------------------
# Extract date components that might influence complaint resolution patterns
# (e.g., complaints on weekends might take longer to resolve)

# Extract day of week (0=Monday, 6=Sunday) from submission date
df['day_of_week'] = pd.to_datetime(df['date_submitted']).dt.dayofweek

# Extract month (1-12) to capture seasonal patterns
df['month'] = pd.to_datetime(df['date_submitted']).dt.month

# Create binary flag for weekend submissions (1=weekend, 0=weekday)
# Hypothesis: Weekend complaints might have different resolution times
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Extract year for trend analysis and model training
df['year'] = pd.to_datetime(df['date_submitted']).dt.year

# -------------------------------------------------------------------------
# 2. HANDLE MISSING VALUES
# -------------------------------------------------------------------------
# Replace NULL values with meaningful defaults to avoid errors during model training

df['state'] = df['state'].fillna('UNKNOWN')  # Unknown location
df['sub_product'] = df['sub_product'].fillna('Not Specified')  # Product detail missing
df['sub_issue'] = df['sub_issue'].fillna('Not Specified')  # Issue detail missing

# -------------------------------------------------------------------------
# 3. CREATE TARGET VARIABLES
# -------------------------------------------------------------------------
# Create binary classification target for monetary relief prediction
# Target = 1 if resolution included monetary relief, 0 otherwise
df['has_monetary_relief'] = df['resolution_status'].str.contains('Monetary Relief', na=False).astype(int)

# -------------------------------------------------------------------------
# 4. ENCODE CATEGORICAL VARIABLES
# -------------------------------------------------------------------------
# Convert text categories to numeric codes that ML algorithms can process
# LabelEncoder assigns each unique category a number (0, 1, 2, ...)

# Initialize encoders for each categorical feature
le_product = LabelEncoder()  # For product categories (Cards, Mortgage, etc.)
le_issue = LabelEncoder()     # For complaint issues
le_state = LabelEncoder()     # For US states
le_channel = LabelEncoder()   # For submission channels (Web, Phone, etc.)
le_resolution = LabelEncoder() # For resolution status types

# Fit encoders and transform categorical columns to numeric
df['product_encoded'] = le_product.fit_transform(df['product_category'])
df['issue_encoded'] = le_issue.fit_transform(df['issue'])
df['state_encoded'] = le_state.fit_transform(df['state'])
df['channel_encoded'] = le_channel.fit_transform(df['submitted_via'])
df['resolution_encoded'] = le_resolution.fit_transform(df['resolution_status'])

# -------------------------------------------------------------------------
# 5. SAVE LABEL ENCODERS FOR LATER USE
# -------------------------------------------------------------------------
# Store encoders in a dictionary so they can be reused for SQL interface
label_encoders = {
    'product': le_product,
    'issue': le_issue,
    'state': le_state,
    'submitted_via': le_channel,  # Note: using 'submitted_via' as key
    'resolution': le_resolution
}

print("\n✓ Label encoders saved to 'label_encoders' dictionary")

# -------------------------------------------------------------------------
# 6. SUMMARY STATISTICS
# -------------------------------------------------------------------------
# Display feature engineering results
print(f"\nFeature engineering complete.")
print(f"Total features created: {len(df.columns)}")
print(f"\nEncoded categories:")
print(f"  - Products: {len(le_product.classes_)} unique categories")
print(f"  - Issues: {len(le_issue.classes_)} unique categories")
print(f"  - States: {len(le_state.classes_)} unique categories")
print(f"  - Channels: {len(le_channel.classes_)} unique categories")
print(f"  - Resolution types: {len(le_resolution.classes_)} unique categories")

In [0]:
# =============================================================================
# MODEL 1: RESOLUTION TIME PREDICTION (REGRESSION)
# =============================================================================
# Goal: Predict how many days it will take to resolve a complaint
# Use Case: Resource planning, SLA management, setting customer expectations
# Algorithm: Gradient Boosting Regressor (ensemble method that builds trees sequentially)

print("\n" + "=" * 60)
print("MODEL 1: RESOLUTION TIME PREDICTION")
print("=" * 60)

# -------------------------------------------------------------------------
# 1. PREPARE FEATURES AND TARGET
# -------------------------------------------------------------------------
# Select features that are known at the time a complaint is submitted
# (cannot use resolution_status since it's only known after resolution)

feature_cols = [
    'product_encoded',      # Which product the complaint is about
    'issue_encoded',        # Type of issue reported
    'state_encoded',        # Geographic location of complaint
    'channel_encoded',      # How complaint was submitted (Web, Phone, etc.)
    'day_of_week',          # Day of week submitted
    'month',                # Month of submission (seasonal patterns)
    'is_weekend',           # Weekend vs. weekday submission
    'complaint_year',       # Year (for trend analysis)
    'complaint_quarter'     # Quarter (for quarterly patterns)
]

# X = Features (independent variables), y = Target (dependent variable)
X_time = df[feature_cols]
y_time = df['response_time_days']  # Target: number of days to resolve

# -------------------------------------------------------------------------
# 2. SPLIT DATA INTO TRAINING AND TEST SETS
# -------------------------------------------------------------------------
# 80% for training the model, 20% for testing (evaluating performance on unseen data)
# random_state=42 ensures reproducibility (same split every time)
X_train, X_test, y_train, y_test = train_test_split(
    X_time, y_time, test_size=0.2, random_state=42
)

print(f"\nTraining set size: {len(X_train)} complaints")
print(f"Test set size: {len(X_test)} complaints")

# -------------------------------------------------------------------------
# 3. TRAIN THE MODEL
# -------------------------------------------------------------------------
print("\nTraining Gradient Boosting Regressor...")

# Initialize the model with hyperparameters
model_time = GradientBoostingRegressor(
    n_estimators=100,      # Number of boosting stages (trees to build)
    learning_rate=0.1,     # Shrinks contribution of each tree (prevents overfitting)
    max_depth=5,           # Maximum depth of each tree (controls complexity)
    random_state=42        # For reproducibility
)

# Fit the model on training data (learn patterns)
model_time.fit(X_train, y_train)

# -------------------------------------------------------------------------
# 4. MAKE PREDICTIONS
# -------------------------------------------------------------------------
# Generate predictions for both training and test sets
y_pred_train = model_time.predict(X_train)  # Predictions on training data
y_pred_test = model_time.predict(X_test)    # Predictions on test data (unseen)

# -------------------------------------------------------------------------
# 5. EVALUATE MODEL PERFORMANCE
# -------------------------------------------------------------------------
# Calculate multiple metrics to assess prediction accuracy

# MAE (Mean Absolute Error): Average absolute difference between predicted and actual
# Lower is better. Interpretation: "On average, predictions are off by X days"
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)

# RMSE (Root Mean Squared Error): Penalizes large errors more than MAE
# Lower is better. More sensitive to outliers than MAE
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

# R² (R-squared): Proportion of variance explained by the model
# Range: 0 to 1 (or negative if very bad). 1 = perfect predictions, 0 = no better than average
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

# -------------------------------------------------------------------------
# 6. DISPLAY RESULTS
# -------------------------------------------------------------------------
print(f"\n=== Model Performance ===")
print(f"Train MAE: {train_mae:.2f} days (avg prediction error on training data)")
print(f"Test MAE: {test_mae:.2f} days (avg prediction error on unseen data)")
print(f"Train RMSE: {train_rmse:.2f} days")
print(f"Test RMSE: {test_rmse:.2f} days")
print(f"Train R²: {train_r2:.4f} (proportion of variance explained on training data)")
print(f"Test R²: {test_r2:.4f} (proportion of variance explained on test data)")
print("\nNote: If test metrics are similar to train metrics, model generalizes well.")
print("      If test metrics are much worse, model may be overfitting.")

# -------------------------------------------------------------------------
# 7. FEATURE IMPORTANCE ANALYSIS
# -------------------------------------------------------------------------
# Identify which features have the most influence on predictions
# Higher importance = feature has more impact on resolution time predictions

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_time.feature_importances_  # Importance scores from the model
}).sort_values('importance', ascending=False)  # Sort by most important first

print("\n=== Feature Importance ===")
print("Features ranked by their impact on resolution time predictions:")
display(importance_df)

In [0]:
# =============================================================================
# SQL INTERFACE: RESOLUTION TIME PREDICTION UDF
# =============================================================================
# Make the Python model callable from SQL queries

import numpy as np
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType
import pandas as pd

print("\n" + "="*60)
print("CREATING SQL INTERFACE FOR MODEL 1")
print("="*60)

# -------------------------------------------------------------------------
# 1. MODEL IS ALREADY IN MEMORY (model_time)
# -------------------------------------------------------------------------
# On serverless compute, we keep models in memory rather than persisting to DBFS
print(f"✓ Using in-memory model: model_time")
print(f"  Model type: {type(model_time).__name__}")
print(f"  Features: {len(feature_cols)} columns")

# -------------------------------------------------------------------------
# 2. CREATE PANDAS UDF FOR BATCH PREDICTIONS
# -------------------------------------------------------------------------
# Pandas UDF is optimized for vectorized operations on Spark DataFrames
# The UDF captures the model through Python closure

@pandas_udf(DoubleType())
def predict_resolution_time(
    product_encoded: pd.Series,
    issue_encoded: pd.Series,
    state_encoded: pd.Series,
    channel_encoded: pd.Series,
    day_of_week: pd.Series,
    month: pd.Series,
    is_weekend: pd.Series,
    complaint_year: pd.Series,
    complaint_quarter: pd.Series
) -> pd.Series:
    """
    Predict resolution time in days for complaints.
    
    Args:
        Multiple encoded features (see Model 1 training cell for details)
    
    Returns:
        Predicted resolution time in days
    """
    # Use the in-memory model (captured via closure)
    model = model_time
    
    # Create feature matrix matching training data format
    X = pd.DataFrame({
        'product_encoded': product_encoded,
        'issue_encoded': issue_encoded,
        'state_encoded': state_encoded,
        'channel_encoded': channel_encoded,
        'day_of_week': day_of_week,
        'month': month,
        'is_weekend': is_weekend,
        'complaint_year': complaint_year,
        'complaint_quarter': complaint_quarter
    })
    
    # Make predictions
    predictions = model.predict(X)
    return pd.Series(predictions)

# Register the UDF so it's available in SQL
spark.udf.register("predict_resolution_time", predict_resolution_time)

print("✓ UDF 'predict_resolution_time' registered for SQL use")
print("\nUsage in SQL:")
print("  SELECT complaint_id, ")
print("         predict_resolution_time(product_encoded, issue_encoded, state_encoded, ")
print("                                 channel_encoded, day_of_week, month, is_weekend, ")
print("                                 complaint_year, complaint_quarter) as predicted_days")
print("  FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints")

In [0]:
# =============================================================================
# PREPARE DATA FOR SQL QUERIES
# =============================================================================
# Create a Spark temp view with encoded features so SQL can use the ML UDFs

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

print("\n" + "="*60)
print("PREPARING DATA FOR SQL PREDICTIONS")
print("="*60)

# Load Silver table as Spark DataFrame
silver_df = spark.table("Bank_of_America.Bank_of_America_Silver.consumer_complaints")

print(f"\nLoaded {silver_df.count()} complaints from Silver layer")

# Convert to pandas for encoding (same as training)
df_for_encoding = silver_df.toPandas()

# Apply the same label encodings used in training
for col in ['product', 'issue', 'state', 'submitted_via']:
    if col in label_encoders:
        # For states, handle any new values not seen during training
        if col == 'state':
            df_for_encoding['state'] = df_for_encoding['state'].fillna('UNKNOWN')
        
        # Encode using the same encoder from training
        encoded_col_name = f"{col}_encoded"
        
        try:
            df_for_encoding[encoded_col_name] = label_encoders[col].transform(df_for_encoding[col])
        except ValueError:
            # Handle unseen categories by assigning them to a default value (0)
            print(f"  Warning: Found new categories in {col}, assigning default encoding")
            df_for_encoding[encoded_col_name] = df_for_encoding[col].apply(
                lambda x: label_encoders[col].transform([x])[0] if x in label_encoders[col].classes_ else 0
            )

# Rename channel encoding to match training
df_for_encoding['channel_encoded'] = df_for_encoding['submitted_via_encoded']

print("✓ Applied label encodings to all categorical features")

# Convert back to Spark DataFrame
encoded_spark_df = spark.createDataFrame(df_for_encoding)

# Create temporary view for SQL to query
encoded_spark_df.createOrReplaceTempView("silver_complaints_with_encodings")

print("✓ Created temp view: silver_complaints_with_encodings")
print("\nSQL queries can now reference this view to access encoded features")
print("Example: SELECT product, product_encoded FROM silver_complaints_with_encodings LIMIT 5")

# Verify the view
print("\nSample of encoded features:")
display(spark.sql("""
    SELECT 
        product, product_encoded,
        issue, issue_encoded,
        state, state_encoded,
        submitted_via, channel_encoded
    FROM silver_complaints_with_encodings 
    LIMIT 5
"""))

In [0]:
%sql
-- =============================================================================
-- GENERATE RESOLUTION TIME PREDICTIONS FOR ALL COMPLAINTS
-- =============================================================================
-- Apply Model 1 to predict resolution time for each complaint in Silver layer
-- Results stored in Gold layer for SQL users, dashboards, and reporting

CREATE OR REPLACE TABLE Bank_of_America.Bank_of_America_Gold.ml_resolution_time_predictions AS
WITH predictions AS (
  SELECT 
      complaint_id,
      product,
      issue,
      state,
      submitted_via,
      date_submitted,
      response_time_days,
      
      -- Call UDF to get prediction
      predict_resolution_time(
          product_encoded,
          issue_encoded,
          state_encoded,
          channel_encoded,
          DAYOFWEEK(date_submitted) - 1,
          MONTH(date_submitted),
          CASE WHEN DAYOFWEEK(date_submitted) IN (1, 7) THEN 1 ELSE 0 END,
          complaint_year,
          complaint_quarter
      ) as predicted_response_time_days
      
  FROM silver_complaints_with_encodings
)
SELECT 
    complaint_id,
    product,
    issue,
    state,
    submitted_via,
    date_submitted,
    response_time_days as actual_response_time_days,
    predicted_response_time_days,
    response_time_days - predicted_response_time_days as prediction_error_days,
    CURRENT_TIMESTAMP() as prediction_generated_at
FROM predictions;

-- Display summary statistics
SELECT 
    COUNT(*) as total_predictions,
    ROUND(AVG(actual_response_time_days), 2) as avg_actual_days,
    ROUND(AVG(predicted_response_time_days), 2) as avg_predicted_days,
    ROUND(AVG(ABS(prediction_error_days)), 2) as avg_absolute_error,
    ROUND(STDDEV(prediction_error_days), 2) as std_error
FROM Bank_of_America.Bank_of_America_Gold.ml_resolution_time_predictions;

In [0]:
# =============================================================================
# MODEL 2: MONETARY RELIEF PREDICTION (BINARY CLASSIFICATION)
# =============================================================================
# Goal: Predict whether a complaint will result in monetary relief to the customer
# Use Case: Cost forecasting, case prioritization, resource allocation
# Algorithm: Logistic Regression (probabilistic classification model)

print("\n" + "=" * 60)
print("MODEL 2: MONETARY RELIEF PREDICTION")
print("=" * 60)

# -------------------------------------------------------------------------
# 1. PREPARE FEATURES AND TARGET
# -------------------------------------------------------------------------
# Select features that might indicate likelihood of monetary relief
# Including response_time_days since faster responses might correlate with relief outcomes

feature_cols_relief = [
    'product_encoded',       # Product type (some products may have higher relief rates)
    'issue_encoded',         # Issue type (certain issues may warrant monetary compensation)
    'state_encoded',         # Geographic location (state regulations vary)
    'channel_encoded',       # Submission channel
    'response_time_days',    # How long it took to resolve (available after resolution)
    'complaint_year'         # Year (policies may change over time)
]

# X = Features, y = Target (binary: 1=monetary relief given, 0=no monetary relief)
X_relief = df[feature_cols_relief]
y_relief = df['has_monetary_relief']  # Binary target created in feature engineering

# -------------------------------------------------------------------------
# 2. SPLIT DATA WITH STRATIFICATION
# -------------------------------------------------------------------------
# stratify=y_relief ensures train and test sets have same proportion of positive/negative classes
# This is important when classes are imbalanced (more "no relief" than "relief" cases)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_relief, y_relief, test_size=0.2, random_state=42, stratify=y_relief
)

# Display class distribution to understand data imbalance
print(f"\nClass distribution in training set:")
print(y_train_r.value_counts(normalize=True))  # Shows percentage of each class
print("0 = No Monetary Relief, 1 = Monetary Relief")

# -------------------------------------------------------------------------
# 3. TRAIN THE MODEL WITH CLASS BALANCING
# -------------------------------------------------------------------------
print("\nTraining Logistic Regression...")

# Initialize Logistic Regression classifier
model_relief = LogisticRegression(
    max_iter=1000,           # Maximum iterations for convergence
    random_state=42,         # For reproducibility
    class_weight='balanced'  # Automatically adjusts weights inversely proportional to class frequencies
                             # This helps the model not to just predict the majority class
)

# Fit the model on training data
model_relief.fit(X_train_r, y_train_r)

# -------------------------------------------------------------------------
# 4. MAKE PREDICTIONS
# -------------------------------------------------------------------------
# Generate class predictions (0 or 1)
y_pred_train_r = model_relief.predict(X_train_r)
y_pred_test_r = model_relief.predict(X_test_r)

# Generate probability predictions (between 0 and 1)
# Useful for ranking cases by likelihood of monetary relief
y_pred_proba_test = model_relief.predict_proba(X_test_r)[:, 1]  # Probability of class 1 (relief)

# -------------------------------------------------------------------------
# 5. EVALUATE MODEL PERFORMANCE
# -------------------------------------------------------------------------
# For classification, we use different metrics than regression

# Accuracy: Proportion of correct predictions (both positive and negative)
# Range: 0 to 1. Higher is better. But can be misleading with imbalanced classes.
train_acc = accuracy_score(y_train_r, y_pred_train_r)
test_acc = accuracy_score(y_test_r, y_pred_test_r)

# AUC-ROC (Area Under Receiver Operating Characteristic Curve)
# Measures model's ability to distinguish between classes across all thresholds
# Range: 0 to 1. 0.5 = random guessing, 1.0 = perfect classification
# Better metric than accuracy for imbalanced datasets
test_auc = roc_auc_score(y_test_r, y_pred_proba_test)

# -------------------------------------------------------------------------
# 6. DISPLAY RESULTS
# -------------------------------------------------------------------------
print(f"\n=== Model Performance ===")
print(f"Train Accuracy: {train_acc:.4f} ({train_acc*100:.2f}% correct predictions)")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}% correct predictions)")
print(f"Test AUC-ROC: {test_auc:.4f} (ability to distinguish relief vs. no relief)")

# -------------------------------------------------------------------------
# 7. DETAILED CLASSIFICATION REPORT
# -------------------------------------------------------------------------
# Shows precision, recall, and F1-score for each class
# Precision: Of all predicted "relief" cases, how many were actually "relief"?
# Recall: Of all actual "relief" cases, how many did we correctly identify?
# F1-score: Harmonic mean of precision and recall (balanced metric)
print("\n=== Classification Report ===")
print("Metrics for each class:")
print("  - Precision: Accuracy of positive predictions")
print("  - Recall: Proportion of actual positives correctly identified")
print("  - F1-score: Harmonic mean of precision and recall\n")
print(classification_report(y_test_r, y_pred_test_r, 
                            target_names=['No Monetary Relief', 'Monetary Relief']))

In [0]:
# =============================================================================
# SQL INTERFACE: MONETARY RELIEF PREDICTION UDF
# =============================================================================
# Make the monetary relief classifier callable from SQL

import numpy as np
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType
import pandas as pd

print("\n" + "="*60)
print("CREATING SQL INTERFACE FOR MODEL 2")
print("="*60)

# -------------------------------------------------------------------------
# 1. MODEL IS ALREADY IN MEMORY (model_relief)
# -------------------------------------------------------------------------
print(f"✓ Using in-memory model: model_relief")
print(f"  Model type: {type(model_relief).__name__}")
print(f"  Features: {len(feature_cols_relief)} columns")

# -------------------------------------------------------------------------
# 2. CREATE PANDAS UDF FOR PROBABILITY PREDICTIONS
# -------------------------------------------------------------------------
# Returns probability of monetary relief (0.0 to 1.0)

@pandas_udf(DoubleType())
def predict_monetary_relief_proba(
    product_encoded: pd.Series,
    issue_encoded: pd.Series,
    state_encoded: pd.Series,
    channel_encoded: pd.Series,
    response_time_days: pd.Series,
    complaint_year: pd.Series
) -> pd.Series:
    """
    Predict probability of monetary relief for complaints.
    
    Args:
        Encoded features (see Model 2 training cell for details)
    
    Returns:
        Probability of monetary relief (0.0 = unlikely, 1.0 = very likely)
    """
    # Use the in-memory model (captured via closure)
    model = model_relief
    
    # Create feature matrix
    X = pd.DataFrame({
        'product_encoded': product_encoded,
        'issue_encoded': issue_encoded,
        'state_encoded': state_encoded,
        'channel_encoded': channel_encoded,
        'response_time_days': response_time_days,
        'complaint_year': complaint_year
    })
    
    # Get probability of positive class (monetary relief)
    probabilities = model.predict_proba(X)[:, 1]
    return pd.Series(probabilities)

# Register for SQL use
spark.udf.register("predict_monetary_relief_proba", predict_monetary_relief_proba)

print("✓ UDF 'predict_monetary_relief_proba' registered for SQL use")
print("\nUsage in SQL:")
print("  SELECT complaint_id, ")
print("         predict_monetary_relief_proba(product_encoded, issue_encoded, state_encoded,")
print("                                       channel_encoded, response_time_days, complaint_year) as relief_probability")
print("  FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints")

In [0]:
%sql
-- =============================================================================
-- GENERATE MONETARY RELIEF PREDICTIONS FOR ALL COMPLAINTS
-- =============================================================================
-- Apply Model 2 to predict likelihood of monetary relief
-- Useful for cost forecasting and case prioritization

CREATE OR REPLACE TABLE Bank_of_America.Bank_of_America_Gold.ml_monetary_relief_predictions AS
WITH predictions AS (
  SELECT 
      complaint_id,
      product,
      issue,
      state,
      company_response_to_consumer,
      date_submitted,
      resolution_status,
      
      -- Actual outcome
      CASE 
          WHEN resolution_status = 'Closed - Monetary Relief' THEN 1 
          ELSE 0 
      END as actual_monetary_relief,
      
      -- Call UDF to get prediction probability
      predict_monetary_relief_proba(
          product_encoded,
          issue_encoded,
          state_encoded,
          channel_encoded,
          response_time_days,
          complaint_year
      ) as predicted_relief_probability
      
  FROM silver_complaints_with_encodings
)
SELECT 
    complaint_id,
    product,
    issue,
    state,
    company_response_to_consumer,
    date_submitted,
    resolution_status,
    actual_monetary_relief,
    predicted_relief_probability,
    CASE 
        WHEN predicted_relief_probability >= 0.5 THEN 1 
        ELSE 0 
    END as predicted_relief_binary,
    CASE 
        WHEN predicted_relief_probability >= 0.7 THEN 'High Risk'
        WHEN predicted_relief_probability >= 0.4 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END as cost_risk_category,
    CURRENT_TIMESTAMP() as prediction_generated_at
FROM predictions;

-- Display model performance on full dataset
SELECT 
    COUNT(*) as total_complaints,
    SUM(actual_monetary_relief) as actual_relief_cases,
    ROUND(SUM(predicted_relief_binary), 0) as predicted_relief_cases,
    ROUND(AVG(predicted_relief_probability) * 100, 2) as avg_relief_probability_pct,
    SUM(CASE WHEN actual_monetary_relief = predicted_relief_binary THEN 1 ELSE 0 END) as correct_predictions,
    ROUND(SUM(CASE WHEN actual_monetary_relief = predicted_relief_binary THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as accuracy_pct
FROM Bank_of_America.Bank_of_America_Gold.ml_monetary_relief_predictions;

In [0]:
# =============================================================================
# MODEL 3: ISSUE CLASSIFICATION (MULTI-CLASS CLASSIFICATION)
# =============================================================================
# Goal: Automatically categorize complaints into issue types based on other features
# Use Case: Automated routing, trend detection, early warning for emerging issues
# Algorithm: Random Forest Classifier (ensemble of decision trees)

print("\n" + "=" * 60)
print("MODEL 3: ISSUE CLASSIFICATION")
print("=" * 60)

# -------------------------------------------------------------------------
# 1. PREPARE DATA - FOCUS ON TOP 10 ISSUES
# -------------------------------------------------------------------------
# To make the problem manageable, we'll classify only the 10 most common issues
# In production, you could expand to all issues or use hierarchical classification

# Get the 10 most frequent issue types
top_issues = df['issue'].value_counts().head(10).index
print(f"\nTop 10 most common issues:")
for i, issue in enumerate(top_issues, 1):
    count = df[df['issue'] == issue].shape[0]
    print(f"  {i}. {issue}: {count} complaints")

# Filter dataset to include only these top 10 issues
df_issues = df[df['issue'].isin(top_issues)].copy()
print(f"\nFiltered dataset size: {len(df_issues)} complaints")

# -------------------------------------------------------------------------
# 2. SELECT FEATURES AND TARGET
# -------------------------------------------------------------------------
# Note: We CANNOT use 'issue_encoded' as a feature since that's what we're trying to predict
# We use features available before the issue is categorized

feature_cols_issue = [
    'product_encoded',      # Product type (different products have different common issues)
    'state_encoded',        # Geographic patterns in issues
    'channel_encoded',      # Submission channel may correlate with issue type
    'complaint_year',       # Issues may evolve over time
    'complaint_month'       # Seasonal patterns in issues
]

X_issue = df_issues[feature_cols_issue]
y_issue = df_issues['issue_encoded']  # Target: encoded issue category (0-9 for top 10)

# -------------------------------------------------------------------------
# 3. SPLIT DATA WITH STRATIFICATION
# -------------------------------------------------------------------------
# Stratification ensures each issue type is proportionally represented in train and test sets
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_issue, y_issue, test_size=0.2, random_state=42, stratify=y_issue
)

print(f"\nClassifying {len(top_issues)} most common issue types")
print(f"Training set size: {len(X_train_i)} complaints")
print(f"Test set size: {len(X_test_i)} complaints")

# -------------------------------------------------------------------------
# 4. TRAIN RANDOM FOREST CLASSIFIER
# -------------------------------------------------------------------------
print("\nTraining Random Forest Classifier...")

# Initialize Random Forest Classifier
model_issue = RandomForestClassifier(
    n_estimators=100,    # Number of decision trees in the forest
                        # More trees = better performance but slower training
    max_depth=10,       # Maximum depth of each tree
                        # Deeper trees can capture more complex patterns but may overfit
    random_state=42,    # For reproducibility
    n_jobs=-1           # Use all available CPU cores for parallel training
)

# Fit the model on training data
# Random Forest builds multiple decision trees and aggregates their predictions
model_issue.fit(X_train_i, y_train_i)

# -------------------------------------------------------------------------
# 5. MAKE PREDICTIONS
# -------------------------------------------------------------------------
# Predict issue categories for train and test sets
y_pred_train_i = model_issue.predict(X_train_i)
y_pred_test_i = model_issue.predict(X_test_i)

# -------------------------------------------------------------------------
# 6. EVALUATE MODEL PERFORMANCE
# -------------------------------------------------------------------------
# For multi-class classification, accuracy tells us the proportion of 
# complaints that were correctly categorized into their issue type

train_acc_i = accuracy_score(y_train_i, y_pred_train_i)
test_acc_i = accuracy_score(y_test_i, y_pred_test_i)

# -------------------------------------------------------------------------
# 7. DISPLAY RESULTS
# -------------------------------------------------------------------------
print(f"\n=== Model Performance ===")
print(f"Train Accuracy: {train_acc_i:.4f} ({train_acc_i*100:.2f}% correctly classified)")
print(f"Test Accuracy: {test_acc_i:.4f} ({test_acc_i*100:.2f}% correctly classified)")
print(f"\nInterpretation: The model correctly predicts the issue category for")
print(f"{test_acc_i*100:.1f}% of unseen complaints based on product, location, channel, and time.")

# -------------------------------------------------------------------------
# 8. FEATURE IMPORTANCE ANALYSIS
# -------------------------------------------------------------------------
# Identify which features are most useful for predicting issue categories
# High importance = feature strongly influences which issue type is predicted

importance_issue_df = pd.DataFrame({
    'feature': feature_cols_issue,
    'importance': model_issue.feature_importances_  # Importance scores from Random Forest
}).sort_values('importance', ascending=False)  # Sort by most important first

print("\n=== Feature Importance ===")
print("Features ranked by their impact on issue classification:")
display(importance_issue_df)

print("\nNote: High importance for 'product_encoded' suggests that knowing the product")
print("is very informative for predicting what type of issue will be reported.")

In [0]:
# =============================================================================
# SQL INTERFACE: ISSUE CLASSIFICATION UDF
# =============================================================================
# Make the issue classifier callable from SQL

import numpy as np
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType, DoubleType
import pandas as pd

print("\n" + "="*60)
print("CREATING SQL INTERFACE FOR MODEL 3")
print("="*60)

# -------------------------------------------------------------------------
# 1. MODELS AND MAPPINGS ARE IN MEMORY
# -------------------------------------------------------------------------
print(f"✓ Using in-memory model: model_issue")
print(f"  Model type: {type(model_issue).__name__}")
print(f"  Features: {len(feature_cols_issue)} columns")
print(f"  Can classify {len(top_issues)} issue types")

# IMPORTANT: Capture the label encoder OUTSIDE the UDF for closure
# This ensures it's available when Spark serializes the UDF
issue_label_encoder = label_encoders['issue']

# -------------------------------------------------------------------------
# 2. CREATE PANDAS UDF FOR ISSUE PREDICTIONS
# -------------------------------------------------------------------------
# Returns the predicted issue category name

@pandas_udf(StringType())
def predict_issue_category(
    product_encoded: pd.Series,
    state_encoded: pd.Series,
    channel_encoded: pd.Series,
    complaint_year: pd.Series,
    complaint_month: pd.Series
) -> pd.Series:
    """
    Predict the issue category for complaints.
    
    Args:
        Encoded features (see Model 3 training cell for details)
    
    Returns:
        Predicted issue category (e.g., 'Managing an account', 'Incorrect information on your report')
    """
    # Use in-memory model and label encoder (captured via closure)
    model = model_issue
    label_encoder = issue_label_encoder  # Use the captured encoder
    
    # Create feature matrix
    X = pd.DataFrame({
        'product_encoded': product_encoded,
        'state_encoded': state_encoded,
        'channel_encoded': channel_encoded,
        'complaint_year': complaint_year,
        'complaint_month': complaint_month
    })
    
    # Predict encoded issue labels
    predictions_encoded = model.predict(X)
    
    # Decode back to original issue names
    predictions = label_encoder.inverse_transform(predictions_encoded)
    
    return pd.Series(predictions)

# Create UDF for confidence scores
@pandas_udf(DoubleType())
def predict_issue_confidence(
    product_encoded: pd.Series,
    state_encoded: pd.Series,
    channel_encoded: pd.Series,
    complaint_year: pd.Series,
    complaint_month: pd.Series
) -> pd.Series:
    """
    Predict confidence score for issue classification.
    
    Returns:
        Maximum probability across all classes (0.0 to 1.0)
        Higher = more confident in the prediction
    """
    # Use in-memory model
    model = model_issue
    
    # Create feature matrix
    X = pd.DataFrame({
        'product_encoded': product_encoded,
        'state_encoded': state_encoded,
        'channel_encoded': channel_encoded,
        'complaint_year': complaint_year,
        'complaint_month': complaint_month
    })
    
    # Get probability for each class and return the maximum
    probabilities = model.predict_proba(X)
    max_probs = probabilities.max(axis=1)
    
    return pd.Series(max_probs)

# Register both UDFs for SQL use
spark.udf.register("predict_issue_category", predict_issue_category)
spark.udf.register("predict_issue_confidence", predict_issue_confidence)

print("✓ UDFs registered for SQL use:")
print("  - predict_issue_category: Returns predicted issue name")
print("  - predict_issue_confidence: Returns prediction confidence (0.0-1.0)")
print("\nUsage in SQL:")
print("  SELECT complaint_id, ")
print("         predict_issue_category(product_encoded, state_encoded, channel_encoded,")
print("                                complaint_year, complaint_month) as predicted_issue,")
print("         predict_issue_confidence(product_encoded, state_encoded, channel_encoded,")
print("                                  complaint_year, complaint_month) as confidence")
print("  FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints")

In [0]:
%sql
-- =============================================================================
-- GENERATE ISSUE CLASSIFICATION PREDICTIONS FOR ALL COMPLAINTS
-- =============================================================================
-- Apply Model 3 to predict issue categories
-- Useful for automated routing and trend detection

CREATE OR REPLACE TABLE Bank_of_America.Bank_of_America_Gold.ml_issue_predictions AS
WITH predictions AS (
  SELECT 
      complaint_id,
      product,
      state,
      submitted_via,
      date_submitted,
      complaint_year,
      complaint_month,
      issue,
      
      -- Call UDFs to get predictions
      predict_issue_category(
          product_encoded,
          state_encoded,
          channel_encoded,
          complaint_year,
          complaint_month
      ) as predicted_issue,
      
      predict_issue_confidence(
          product_encoded,
          state_encoded,
          channel_encoded,
          complaint_year,
          complaint_month
      ) as prediction_confidence
      
  FROM silver_complaints_with_encodings
)
SELECT 
    complaint_id,
    product,
    state,
    submitted_via,
    date_submitted,
    complaint_year,
    complaint_month,
    issue as actual_issue,
    predicted_issue,
    prediction_confidence,
    CASE 
        WHEN prediction_confidence >= 0.8 THEN 'High Confidence'
        WHEN prediction_confidence >= 0.5 THEN 'Medium Confidence'
        ELSE 'Low Confidence - Manual Review Needed'
    END as confidence_category,
    CASE 
        WHEN issue = predicted_issue THEN 1 
        ELSE 0 
    END as is_correct_prediction,
    CURRENT_TIMESTAMP() as prediction_generated_at
FROM predictions;

-- Display model performance summary
SELECT 
    COUNT(*) as total_predictions,
    COUNT(DISTINCT predicted_issue) as unique_predicted_issues,
    ROUND(AVG(prediction_confidence) * 100, 2) as avg_confidence_pct,
    SUM(is_correct_prediction) as correct_predictions,
    ROUND(SUM(is_correct_prediction) * 100.0 / COUNT(*), 2) as accuracy_pct,
    SUM(CASE WHEN confidence_category = 'High Confidence' THEN 1 ELSE 0 END) as high_confidence_count,
    ROUND(SUM(CASE WHEN confidence_category = 'High Confidence' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as high_confidence_pct
FROM Bank_of_America.Bank_of_America_Gold.ml_issue_predictions;

# SQL Interface Usage Guide

## Overview

This notebook provides **hybrid Python + SQL ML capabilities**:

* **Python Models** (Cells 3-6): Train accurate ML models using scikit-learn
* **SQL Interfaces** (New cells): Make predictions accessible via SQL queries
* **Gold Tables**: Store prediction results for easy querying

---

## Available SQL Functions

### 1. Resolution Time Prediction
```sql
predict_resolution_time(
    product_encoded, issue_encoded, state_encoded, 
    channel_encoded, day_of_week, month, is_weekend, 
    complaint_year, complaint_quarter
) → DOUBLE (predicted days)
```

### 2. Monetary Relief Prediction
```sql
predict_monetary_relief_proba(
    product_encoded, issue_encoded, state_encoded,
    channel_encoded, response_time_days, complaint_year
) → DOUBLE (probability 0.0-1.0)
```

### 3. Issue Classification
```sql
predict_issue_category(
    product_encoded, state_encoded, channel_encoded,
    complaint_year, complaint_month
) → STRING (issue category)

predict_issue_confidence(
    product_encoded, state_encoded, channel_encoded,
    complaint_year, complaint_month
) → DOUBLE (confidence 0.0-1.0)
```

---

## Gold Tables Created

1. **`Bank_of_America.Bank_of_America_Gold.ml_resolution_time_predictions`**
   - Predicted resolution time for all complaints
   - Includes actual vs predicted comparison
   - Use for: SLA forecasting, resource planning

2. **`Bank_of_America.Bank_of_America_Gold.ml_monetary_relief_predictions`**
   - Probability and risk category for monetary relief
   - Use for: Cost forecasting, case prioritization

3. **`Bank_of_America.Bank_of_America_Gold.ml_issue_predictions`**
   - Predicted issue categories with confidence scores
   - Use for: Automated routing, trend detection

---

## Example SQL Queries for Business Users

### Query 1: High-Risk Cases for Cost Planning
```sql
SELECT 
    product,
    state,
    COUNT(*) as complaint_count,
    AVG(predicted_relief_probability) as avg_relief_probability,
    SUM(CASE WHEN cost_risk_category = 'High Risk' THEN 1 ELSE 0 END) as high_risk_count
FROM Bank_of_America.Bank_of_America_Gold.ml_monetary_relief_predictions
WHERE complaint_year = 2024
GROUP BY product, state
HAVING high_risk_count > 10
ORDER BY avg_relief_probability DESC;
```

### Query 2: SLA Monitoring - Cases Likely to Miss Target
```sql
SELECT 
    complaint_id,
    product,
    issue,
    date_submitted,
    predicted_response_time_days,
    CASE 
        WHEN predicted_response_time_days > 3 THEN 'At Risk'
        WHEN predicted_response_time_days > 2 THEN 'Monitor'
        ELSE 'On Track'
    END as sla_status
FROM Bank_of_America.Bank_of_America_Gold.ml_resolution_time_predictions
WHERE date_submitted >= CURRENT_DATE() - INTERVAL 30 DAYS
  AND predicted_response_time_days > 2
ORDER BY predicted_response_time_days DESC;
```

### Query 3: Emerging Issue Trends
```sql
SELECT 
    complaint_year,
    complaint_month,
    predicted_issue,
    COUNT(*) as issue_count,
    AVG(prediction_confidence) as avg_confidence
FROM Bank_of_America.Bank_of_America_Gold.ml_issue_predictions
WHERE confidence_category IN ('High Confidence', 'Medium Confidence')
GROUP BY complaint_year, complaint_month, predicted_issue
HAVING issue_count > 50
ORDER BY complaint_year DESC, complaint_month DESC, issue_count DESC;
```

### Query 4: Model Performance Dashboard
```sql
-- Combined metrics for executive dashboard
SELECT 
    'Resolution Time' as model,
    ROUND(AVG(ABS(prediction_error_days)), 2) as avg_error,
    'days' as unit
FROM Bank_of_America.Bank_of_America_Gold.ml_resolution_time_predictions

UNION ALL

SELECT 
    'Monetary Relief' as model,
    ROUND(AVG(CASE WHEN actual_monetary_relief = predicted_relief_binary THEN 100 ELSE 0 END), 2) as avg_error,
    '% accuracy' as unit
FROM Bank_of_America.Bank_of_America_Gold.ml_monetary_relief_predictions

UNION ALL

SELECT 
    'Issue Classification' as model,
    ROUND(AVG(is_correct_prediction) * 100, 2) as avg_error,
    '% accuracy' as unit
FROM Bank_of_America.Bank_of_America_Gold.ml_issue_predictions;
```

---

## Workflow

### For Data Scientists:
1. Run Python cells (1-6) to train/retrain models
2. Run SQL Interface cells to create UDFs and Gold tables
3. Models are persisted to `/dbfs/FileStore/models/`

### For Business Users:
1. Query Gold tables directly (no need to run Python)
2. Use SQL functions in your own queries
3. Build dashboards on top of prediction tables
4. Set up alerts based on risk categories

### For Scheduling:
* Use Databricks Jobs to run this notebook weekly/monthly
* Predictions in Gold tables automatically refresh
* Downstream dashboards and reports stay current

---

## Next Steps

* **Dashboard Integration**: Create Lakeview dashboards using Gold tables
* **Alerting**: Set up SQL alerts for high-risk cases
* **Model Monitoring**: Track prediction accuracy over time
* **Feature Updates**: Add new features to improve model accuracy
* **A/B Testing**: Compare model predictions against business rules

## Model Deployment

All models are logged to MLflow and can be:
1. **Registered** to Unity Catalog Model Registry
2. **Deployed** as Model Serving endpoints
3. **Integrated** into the ETL pipeline for real-time predictions

---

## Using the Models

### Resolution Time Prediction
```python
# Load model
model_uri = "runs:/<run_id>/model"
loaded_model = mlflow.sklearn.load_model(model_uri)

# Predict
new_complaint = [[product_id, issue_id, state_id, channel_id, day, month, is_weekend, year, quarter]]
predicted_days = loaded_model.predict(new_complaint)
```

### Monetary Relief Prediction
```python
# Predict probability of monetary relief
relief_probability = model_relief.predict_proba(features)[:, 1]
```

### Issue Classification
```python
# Predict issue category
predicted_issue_id = model_issue.predict(features)
predicted_issue = le_issue.inverse_transform(predicted_issue_id)
```

---

## Next Steps

1. **Hyperparameter Tuning**: Use GridSearchCV or Optuna for optimization
2. **Feature Engineering**: Add text features from complaint descriptions
3. **Model Registry**: Register best models to Unity Catalog
4. **Batch Scoring**: Create scheduled job to score new complaints
5. **Model Monitoring**: Track prediction drift and retrain as needed
6. **A/B Testing**: Compare model predictions with actual outcomes